# Explainability and Error Analysis

This notebook explains how the trained model is making decisions.

What this notebook does:
- Load the best saved model.
- Run predictions on the test set.
- Identify correct and incorrect predictions.
- Generate Grad-CAM heatmaps.
- Optionally evaluate Dice and IoU if you later add real tumor masks.

### Code Cell: Setup and Load Trained Model

This code cell imports the required libraries, defines project paths, and loads the trained model that will be explained.

In [ ]:
# Setup: import libraries and load the saved trained model.
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

# Use the notebook location as the project root.
PROJECT_ROOT = Path.cwd()
DATASET_ROOT = PROJECT_ROOT / 'Dataset' / 'split_data'
MODELS_ROOT = PROJECT_ROOT / 'artifacts' / 'models'
EXPLAINABILITY_ROOT = PROJECT_ROOT / 'artifacts' / 'explainability'
EXPLAINABILITY_ROOT.mkdir(parents=True, exist_ok=True)

CLASSES = ['glioma', 'healthy', 'meningioma', 'pituitary']
IMAGE_SIZE = (224, 224)

# Prefer the stronger transfer-learning model if it has already been trained.
model_path = MODELS_ROOT / 'efficientnet_b0.keras'
if not model_path.exists():
    model_path = MODELS_ROOT / 'custom_cnn.keras'

print('Loading model from:', model_path)
model = tf.keras.models.load_model(model_path)

## Step 1: Build a Test Prediction Table

We first create a dataframe of test images so we can inspect predictions file by file. This is helpful for error analysis, especially when we want to trace where the model is making confident mistakes.

### Code Cell: Create the Prediction Dataframe

This cell builds a row-by-row table of test images and defines helper functions for loading images for model inference and display.

In [ ]:
# Step 1: build a dataframe describing every test image.
def build_prediction_frame(split_name='test'):
    # Create one row per image so later analysis can stay tabular.
    rows = []
    valid_suffixes = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

    for class_index, class_name in enumerate(CLASSES):
        class_dir = DATASET_ROOT / split_name / class_name
        for file_path in sorted(class_dir.iterdir()):
            if file_path.is_file() and file_path.suffix.lower() in valid_suffixes:
                rows.append(
                    {
                        'filepath': str(file_path),
                        'filename': file_path.name,
                        'class_name': class_name,
                        'class_index': class_index,
                    }
                )

    frame = pd.DataFrame(rows)
    return frame


def load_image_tensor(image_path):
    # Load an image in float format for model inference and Grad-CAM.
    image_bytes = tf.io.read_file(str(image_path))
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32)
    return tf.expand_dims(image, axis=0)


def load_image_display(image_path):
    # Load the same image in uint8 format for plotting in matplotlib.
    image_bytes = tf.io.read_file(str(image_path))
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.uint8)
    return image.numpy()


prediction_frame = build_prediction_frame('test')
prediction_frame.head()

## Step 2: Run Test Predictions

This section computes predicted class labels and confidence scores. After this, we can separate correct predictions from misclassifications and inspect patterns.

### Code Cell: Run Inference on the Test Set

This cell performs prediction on all test images and appends predicted labels, class names, and confidence scores to the dataframe.

In [ ]:
# Step 2: run model predictions and attach them to the dataframe.
prediction_inputs = []
for image_path in prediction_frame['filepath']:
    # Store resized tensors in a NumPy array for batch prediction.
    prediction_inputs.append(load_image_tensor(image_path)[0].numpy())

prediction_inputs = np.asarray(prediction_inputs, dtype=np.float32)
probabilities = model.predict(prediction_inputs, verbose=0)
prediction_frame['predicted_class_index'] = probabilities.argmax(axis=1)
prediction_frame['predicted_class_name'] = prediction_frame['predicted_class_index'].map(lambda idx: CLASSES[int(idx)])
prediction_frame['predicted_probability'] = probabilities.max(axis=1)
prediction_frame['correct_prediction'] = prediction_frame['class_index'] == prediction_frame['predicted_class_index']

prediction_frame.head()

## Step 3: Review Incorrect Predictions

These rows are often the most useful part of explainability work. They show which classes are being confused and whether the mistakes happen at high confidence.

### Code Cell: Filter Misclassified Examples

This cell isolates the errors so you can inspect the most important misclassifications first.

In [ ]:
# Step 3: isolate incorrect predictions for focused error analysis.
error_frame = prediction_frame.loc[~prediction_frame['correct_prediction']].sort_values(
    by='predicted_probability',
    ascending=False,
)

print('Total test images:', len(prediction_frame))
print('Correct predictions:', int(prediction_frame['correct_prediction'].sum()))
print('Incorrect predictions:', len(error_frame))
display(error_frame.head(10))

## Step 4: Define Grad-CAM Helpers

Grad-CAM helps visualize which image regions contributed most to the final prediction. The helper functions below identify the last convolution layer, generate the heatmap, and overlay it on the original MRI image.

### Code Cell: Define Grad-CAM Utility Functions

This code cell contains the functions used to find the last convolution layer, compute the Grad-CAM heatmap, and overlay the heatmap on the original image.

In [ ]:
# Step 4: define Grad-CAM helper functions.
def find_last_conv_layer(current_model):
    # Search both top-level layers and nested backbone layers.
    for layer in reversed(current_model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            return None, layer.name
        if isinstance(layer, tf.keras.Model):
            for nested_layer in reversed(layer.layers):
                if isinstance(nested_layer, tf.keras.layers.Conv2D):
                    return layer.name, nested_layer.name
    raise ValueError('No convolutional layer found.')


def make_gradcam_heatmap(image_tensor, class_index=None):
    # Build a gradient path from the target class score to the last conv feature map.
    nested_model_name, last_conv_layer_name = find_last_conv_layer(model)

    if nested_model_name is None:
        conv_output_tensor = model.get_layer(last_conv_layer_name).output
    else:
        nested_model = model.get_layer(nested_model_name)
        conv_output_tensor = nested_model.get_layer(last_conv_layer_name).output

    grad_model = tf.keras.Model(inputs=model.inputs, outputs=[conv_output_tensor, model.output])

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(image_tensor, training=False)
        if class_index is None:
            class_index = int(tf.argmax(predictions[0]))
        class_channel = predictions[:, class_index]

    gradients = tape.gradient(class_channel, conv_outputs)
    pooled_gradients = tf.reduce_mean(gradients, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(conv_outputs * pooled_gradients, axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    max_value = tf.reduce_max(heatmap)
    if float(max_value) == 0.0:
        return np.zeros_like(heatmap.numpy())
    heatmap = heatmap / max_value
    return heatmap.numpy()


def overlay_heatmap(display_image, heatmap, alpha=0.4):
    # Resize the heatmap to the original image and blend the two together.
    heatmap = cv2.resize(heatmap, (display_image.shape[1], display_image.shape[0]))
    heatmap_uint8 = np.uint8(255 * np.clip(heatmap, 0, 1))
    colored = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    colored = cv2.cvtColor(colored, cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(display_image, 1 - alpha, colored, alpha, 0)

## Step 5: Visualize Grad-CAM Examples

This section mixes one example from each class with a few errors so you can see both successes and failure cases. That usually gives a more honest view of model behavior than showing only the best examples.

### Code Cell: Generate Grad-CAM Visualizations

This cell creates the visual explanation plots and saves a combined image for later use in reports or the dashboard.

In [ ]:
# Step 5: generate Grad-CAM visualizations for representative examples.
sample_frame = pd.concat(
    [
        prediction_frame.groupby('class_name', group_keys=False).head(1),
        error_frame.head(2),
    ],
    ignore_index=True,
).drop_duplicates(subset=['filepath'])

rows = min(6, len(sample_frame))
fig, axes = plt.subplots(rows, 3, figsize=(12, 4 * rows))
if rows == 1:
    axes = np.expand_dims(axes, axis=0)

for row_index, (_, row) in enumerate(sample_frame.head(rows).iterrows()):
    display_image = load_image_display(row['filepath'])
    image_tensor = load_image_tensor(row['filepath'])
    # Generate the Grad-CAM heatmap for the class predicted by the model.
    heatmap = make_gradcam_heatmap(image_tensor, class_index=int(row['predicted_class_index']))
    overlay = overlay_heatmap(display_image, heatmap)

    axes[row_index, 0].imshow(display_image)
    axes[row_index, 0].set_title(f"Original\nTrue: {row['class_name']}")
    axes[row_index, 1].imshow(heatmap, cmap='jet')
    axes[row_index, 1].set_title(f"Grad-CAM\nPred: {row['predicted_class_name']}")
    axes[row_index, 2].imshow(overlay)
    axes[row_index, 2].set_title(f"Overlay\nConfidence: {row['predicted_probability']:.3f}")

    for axis in axes[row_index]:
        axis.axis('off')

plt.tight_layout()
plt.savefig(EXPLAINABILITY_ROOT / 'gradcam_examples.png', dpi=150)
plt.show()

print('Saved Grad-CAM visualization to', EXPLAINABILITY_ROOT / 'gradcam_examples.png')

## Step 6: Summarize High-Confidence Errors

High-confidence mistakes deserve special attention because they can indicate systematic confusion between tumor subtypes or background artifacts that the model has learned incorrectly.

### Code Cell: Summarize Error Patterns

This cell highlights confident errors and shows which true classes are most often confused with which predicted classes.

In [ ]:
# Step 6: summarize high-confidence errors and confusion patterns.
high_confidence_errors = error_frame.loc[error_frame['predicted_probability'] >= 0.80]
display(high_confidence_errors.head(15))

error_summary = (
    error_frame.groupby(['class_name', 'predicted_class_name'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)
display(error_summary.head(20))

## Optional Dice and IoU

Only run the next cell if you have real tumor masks saved as:
`Dataset/tumor_masks/<class_name>/<filename>`

Without real masks, Dice and IoU are not valid clinical overlap metrics.

### Code Cell: Optional Mask-Overlap Evaluation

Run this only if you have real binary tumor masks. It compares Grad-CAM-derived regions with true masks to estimate Dice and IoU.

In [ ]:
# Step 7: optionally compute Dice and IoU if real tumor masks are available.
MASK_ROOT = PROJECT_ROOT / 'Dataset' / 'tumor_masks'

dice_scores = []
iou_scores = []

if MASK_ROOT.exists():
    for _, row in prediction_frame.head(50).iterrows():
        mask_path = MASK_ROOT / row['class_name'] / row['filename']
        if not mask_path.exists():
            continue

        display_image = load_image_display(row['filepath'])
        image_tensor = load_image_tensor(row['filepath'])
        heatmap = make_gradcam_heatmap(image_tensor, class_index=int(row['predicted_class_index']))
        heatmap = cv2.resize(heatmap, (display_image.shape[1], display_image.shape[0]))
        pred_mask = (heatmap >= 0.5).astype(np.uint8)

        true_mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if true_mask is None:
            continue
        true_mask = cv2.resize(true_mask, (display_image.shape[1], display_image.shape[0]))
        true_mask = (true_mask > 0).astype(np.uint8)

        intersection = np.logical_and(pred_mask, true_mask).sum()
        union = np.logical_or(pred_mask, true_mask).sum()
        dice = (2 * intersection) / (pred_mask.sum() + true_mask.sum()) if (pred_mask.sum() + true_mask.sum()) else 0.0
        iou = intersection / union if union else 0.0

        dice_scores.append(float(dice))
        iou_scores.append(float(iou))

    if dice_scores:
        print('Mean Dice:', np.mean(dice_scores))
        print('Mean IoU:', np.mean(iou_scores))
    else:
        print('Mask folder exists, but no matching masks were found.')
else:
    print('No tumor mask folder found. Dice and IoU skipped.')